# 02 — Baseline network on the withheld-terrain surface

Creates the run dir (`output_data/corridors_wolverine/v2_runNNN/`) on the **headline variant**
(`config.CORRIDORS["wolverine"]["headline_variant"]`, passed as `overrides` so the published surface
stays the config baseline), then: **G0** (node count + naming-file hash), resistance report, the
cost-weighted-distance stage (one pass per node — the expensive step, cached + resumable),
the INHERITED band cutoff (**W5**), the network (**G3**), **GW4** (band store exactness), **G15**,
the linkage-priority surface and `write_run`.

Runs ONLY the variant surface (W10). Sensitivities and audits are parked (spec §7).

In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/wolverine_refugia_connectivity/, below the repo root where
# config.py and the corridor engine modules sit. Same bootstrap as analyses/northern_connectivity/.
import sys, pathlib, json
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import numpy as np
import pandas as pd
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
for _m in (config, cp, cg, cc):
    importlib.reload(_m)

KEY = "wolverine"
CFG = config.CORRIDORS[KEY]
HERE = ROOT / "analyses" / "wolverine_refugia_connectivity"

## 1 · Preconditions — **G8** (config validity) · **G2 / GW1** (warps)

In [ ]:
cfg0, cost0 = cc.resolve(KEY)
g = cp.grid(KEY); cp.check(g); cp.check_classes(g, KEY)
VARIANT = CFG["headline_variant"]
OV = CFG["variants"][VARIANT]
print("headline variant:", OV["resistance"]["variant"]["label"], "->", OV["resistance"]["out_name"])
assert (g["dir"] / OV["resistance"]["out_name"]).exists(), "variant surface missing -- run notebook 01 section 6"

## 2 · Run dir on the variant surface

`require_review=False`: raster patches are single components, there is no H7 review (W7). The
run copies + hash-pins `node_parts.*` and `node_names.csv`, records the variant meta, the class
raster and the git SHA in `run_config.json` — the run dir is the only record that survives.

In [ ]:
A = cc.start(KEY, overrides=OV, label=f"headline: {OV['resistance']['variant']['label']}", require_review=False)
assert A.rec["inputs"]["movement_cost"]["path"].endswith(OV["resistance"]["out_name"]), "the run is not on the variant surface"
assert "resistance_variant" in A.rec["inputs"], "variant meta not recorded"
print("run on", A.rec["inputs"]["movement_cost"]["path"], "| variant meta withheld km²:", A.rec["inputs"]["resistance_variant"]["meta"]["withheld_total_km2"])

## 3 · **G0** — the frozen node set (count from `refugia_summary.json`) + the naming-file hash

In [ ]:
EXPECT_NODES = json.loads((HERE / "audit" / "audit_objects" / "refugia_summary.json").read_text())["n_nodes"]
cc.gate_g0(A, expect_names=EXPECT_NODES, expect_merges=0)

## 4 · Resistance — the variant surface as the run sees it

In [ ]:
cc.resistance(A)
cc.resistance_report(A)

## 5 · Cost-weighted distance from every node — THE expensive stage

One `find_costs` per node (66 × the probe time from notebook 01). Compact cache
(`cwd_compact`, routable cells only) under `input_data/corridors_300m_y2y/cwd_cache/<sha>_c/`;
resumable — re-running the cell skips the parts already on disk.

In [ ]:
cc.cost_distances(A)

## 6 · The band cutoff is INHERITED (**W5**) — never calibrated here

Same 300 m grid and cost units as the north ⇒ the same ~4 km detour allowance. `set_cutoff` pins
it into `run_config.json`. The MST-only new-land area at that cutoff is reported for the record
(the north's R4 analogue); its bands go into the band store and are reused by the network build.

In [ ]:
cc.set_cutoff(A, A.cfg["cwd_cutoff_abs"])
A.D = cc._unit_D(A)
_, mst = cg.build(A.D, [l for l, _ in A.nodes], A.kinds, beta=0, verbose=False)
bands, _, _, _ = cc.edge_bands(A, A.cwd, A.mcp, mst, A.cfg["cwd_cutoff_abs"], "abs", want_slack=False, tag=A.cwd_tag)
corr_mst, _ = cc.network_mask(A, bands)
print(f"MST-only network at the inherited cutoff: {len(mst)} edges, {int(corr_mst.sum())*A.cell_km2:,.0f} km² of new corridor land "
      f"(north/v2_run002: 18,150 km² on 41 edges)")

## 7 · The network — locked MST + β-backup augmentation, per-edge bands, masks (**G3**)

In [ ]:
cc.corridor_network(A)

## 8 · **GW4** — the band store + early-stop traceback reproduce a fresh full-field computation exactly

In [ ]:
cc.gate_gw4(A, n=3)

## 9 · **G15** — centralities finite; current-flow vs shortest-path proportional on the β = 0 tree

In [ ]:
cc.gate_g15(A)

## 9b · Protection status (**W11**) — which links are already connected within PAs / IPCAs

Protection is a STATUS, not a movement cost (D5): routing is untouched. Per link, the share of its
least-cost line inside nodes + existing PAs, and inside nodes + PAs + proposed IPCAs (taken as given);
a link at ≥ 0.95 is "already connected" — by existing PAs alone, or only once the IPCAs are realized.
The corridor land still to secure = the bands outside protected land (`corridors_unprotected.tif`).

In [ ]:
cc.secured_status(A)

## 10 · Criticality — the irreplaceable links (D7) at a glance

In [ ]:
cols = ["label_i", "label_j", "cost", "in_mst", "irreplaceable", "n_pairs_lost", "backup_ratio", "ecfb_norm", "band_km2", "band_unprotected_km2", "secured_by"]
pd.set_option("display.width", 220, "display.max_colwidth", 40)
A.edges[A.edges.is_adjacency == False].sort_values(["irreplaceable", "n_pairs_lost", "ecfb_raw"], ascending=False)[cols].head(25)

## 11 · The linkage-priority surface (D9) + the engine map

In [ ]:
cc.priority_surface(A)
cc.map(A)

## 12 · Write the run

In [ ]:
cc.write_run(A)
cc.runs(KEY)

## Done — next: `03_products.ipynb`

Re-attaches to this run (the band store makes that seconds, not hours), adds the near-optimality
surface (D11, **G10**) and the route branches (D12, **G9**), and finishes the run.